In [ ]:
# Mount Drive and set data path
# Colab setup: mount Drive, set project root, add src to path
import sys, os

# Install deps if running on Colab
if 'google.colab' in sys.modules:
    try:
        import torch, torchvision, cv2  # noqa: F401
    except Exception:
        %pip -q install torch torchvision opencv-python tqdm
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AML-ETH-Project2'
else:
    # Fallback for local runs
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    %load_ext autoreload
    %autoreload 2

SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print('Project root:', PROJECT_ROOT)
print('Using src path:', SRC_PATH)
# Clear cached modules to force reimport of updated code
import sys
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['utils', 'dataset', 'models', 'train', 'UNet']):
        del sys.modules[mod]

print("✓ Module cache cleared - fresh import will happen next")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/AML-ETH-Project2
Using src path: /content/drive/MyDrive/AML-ETH-Project2/src
✓ Module cache cleared - fresh import will happen next


In [ ]:
TEST_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw','test.pkl')
MODEL_PATH = os.path.join(PROJECT_ROOT, 'weights', 'unet_augmented_sub1.pth')

In [ ]:
# define function to get predictions from test data
# Returns a list of dicts with keys:
#   'video_name': str
#   'masks' np.ndarray of shape (T, H, W), uint8
from tqdm import tqdm
import numpy as np
import torch
import cv2

def get_test_predictions(model, data, device, batch_size=32, temp_window=0, target_hw=(256, 256)):
    model.eval()
    results = []

    with torch.no_grad():
        for sample in tqdm(iter(data), desc='Predicting on test data'):
            video = sample['video']         # (H0, W0, T)
            video_name = sample['name']
            H0, W0, T = video.shape

            # Resize all frames to model input size with cv2 (INTER_LINEAR) and normalize
            resized = np.stack(
                [cv2.resize(video[:, :, t], (target_hw[1], target_hw[0]), interpolation=cv2.INTER_LINEAR)
                 for t in range(T)],
                axis=0
            )  # (T, target_h, target_w)
            frames_res = torch.from_numpy(resized.astype(np.float32) / 255.0)[:, None].to(device, non_blocking=True)

            # Batched forward passes to collect logits
            logits_chunks = []
            for s in range(0, T, batch_size):
                e = min(T, s + batch_size)
                logits_chunks.append(model(frames_res[s:e]))
            logits_all = torch.cat(logits_chunks, dim=0)              # (T, 2, 256, 256)
            masks_all = torch.argmax(logits_all, dim=1).cpu().numpy() # (T, 256, 256)

            # Resize masks back to original size with nearest-neighbor
            masks_orig = np.stack(
                [cv2.resize(m, (W0, H0), interpolation=cv2.INTER_NEAREST) for m in masks_all],
                axis=0
            ).astype(np.uint8)
            masks_orig = masks_orig.transpose(1, 2, 0)  # (H0, W0, T)
            results.append({'video_name': video_name, 'masks': masks_orig})
    return results

In [ ]:
# Load test data and model
from utils.utilities import load_zipped_pickle
from models.UNet import UNet
test_data = load_zipped_pickle(TEST_PATH)
print("Data succesfully loadede")
model = UNet(n_channels=1, n_classes=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
print("Model succesfully loaded from", MODEL_PATH)
model.to(device)
preds = get_test_predictions(model, test_data, device)

Data succesfully loadede
Model succesfully loaded from /content/drive/MyDrive/AML-ETH-Project2/weights/unet_augmented_sub1.pth


Predicting on test data: 20it [00:36,  1.85s/it]


In [ ]:
print(preds[0]['video_name'])
print(preds[0]['masks'].shape)
print(np.unique(preds[0]['masks'][...,0]))
print(len(preds[0]['masks'].flatten()))
print(586* 821* 103)

E9AHVWGBUF
(586, 821, 103)
[0 1]
49553918
49553918


In [ ]:
# get sequences for submission
def get_sequences(arr):
    first_indices, last_indices, lengths = [], [], []
    n, i = len(arr), 0
    arr = np.array(arr).astype(np.int32)
    arr = np.concatenate([[0], arr, [0]])
    for index, value in enumerate(arr[:-1]):
        if arr[index+1]-arr[index] == 1:
            first_indices.append(index)
        if arr[index+1]-arr[index] == -1:
            last_indices.append(index)
    lengths = list(np.array(last_indices)-np.array(first_indices))
    return first_indices, lengths

In [ ]:
arr = [0,0,0,1,1,1,0,0,0,1,0,1,0]
get_sequences(arr)

([3, 9, 11], [np.int64(3), np.int64(1), np.int64(1)])

In [ ]:
def get_submission_from_predictions(preds):
    results = []
    for pred in preds:
        name = pred['video_name']
        masks = pred['masks']  # (T, H, W), uint8
        arr = masks.flatten()
        first_indices, lengths = get_sequences(arr)
        for i, (first_index, length) in enumerate(zip(first_indices, lengths)):
            results.append({'id': f"{name}_{i}", 'value': [first_index, length]})
    return results

In [ ]:
submission_list = get_submission_from_predictions(preds)
print(submission_list[:5])

[{'id': 'E9AHVWGBUF_0', 'value': [17887108, np.int64(1)]}, {'id': 'E9AHVWGBUF_1', 'value': [17887211, np.int64(1)]}, {'id': 'E9AHVWGBUF_2', 'value': [17887314, np.int64(1)]}, {'id': 'E9AHVWGBUF_3', 'value': [17887417, np.int64(1)]}, {'id': 'E9AHVWGBUF_4', 'value': [17887520, np.int64(1)]}]


In [54]:
# Save submission to CSV
import pandas as pd

df = pd.DataFrame(submission_list)
df['value'] = df['value'].apply(lambda x: f"[{x[0]}, {x[1]}]")  # Format as "[start, length]"

output_path = os.path.join(PROJECT_ROOT, 'submissions', 'sub1.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"✓ Submission saved to {output_path}")

✓ Submission saved to /content/drive/MyDrive/AML-ETH-Project2/submissions/sub1.csv
